# Explore the XJTU-SY small subset with ISA-PHM

This notebook opens your existing local Wizard export, explores its studies and sensors, reads one raw measurement, and plots it. It does not modify the dataset. The CSV files are not included in this repository.

**Setup:** from the wrapper repository, run `git submodule update --init --recursive`, then `python -m pip install -e ".[notebooks]"` and `jupyter lab`. Select the kernel from that environment and run the cells in order. Change the path below if your dataset is elsewhere.

In [ ]:
from pathlib import Path
import pandas as pd
from isa_phm import ISAWrapper
import numpy as np

# Set this to the extracted XJTU small-subset dataset directory.
dataset_root = Path('path/to/xjtu-small-subset/Dataset').expanduser()
isa_json = dataset_root / 'xjtu-sy-bearing-datasets-small-subset.isa-phm.json'
if not isa_json.is_file():
    raise FileNotFoundError(f'Edit dataset_root above to point to your extracted dataset: {isa_json}')
print('Dataset:', dataset_root)
print('ISA-PHM JSON:', isa_json)

## Open the investigation

Strict ISA validation is disabled here so that this is an exploration example, not a conformance check. Use the specification validator separately for the checks it supports.

In [ ]:
wrapper = ISAWrapper(isa_json, data_root=dataset_root, strict_validation=False)
wrapper.summary()

In [ ]:
pd.DataFrame([study.model_dump() for study in wrapper.list_studies()])

## Explore images and datasheets

These are attachment references from the ISA-PHM export. Each entry retains its study and associated testbed, component, sensor or device. `available` means the referenced local file exists; it does not validate its contents. Notebook previews embed file content in saved output, so clear outputs before sharing if attachments are sensitive. PDF preview depends on your browser.

In [ ]:
attachments = wrapper.attachments(study=1)
display(pd.DataFrame([{'kind': a.kind, 'owner_type': a.owner_type, 'owner': a.owner, 'status': a.status, 'path': str(a.path) if a.path else a.reference} for a in attachments]))

In [ ]:
image = next((a for a in attachments if a.kind == 'image' and a.status == 'available'), None)
if image:
    image.display()
else:
    print('No available image found.')

In [ ]:
datasheet = next((a for a in attachments if a.kind == 'datasheet' and a.status == 'available'), None)
if datasheet:
    print('Datasheet:', datasheet.path)
    datasheet.display()  # If your browser blocks the preview, open the path locally.
else:
    print('No available datasheet found.')

## Choose a bearing and sensor

This export has two bearing studies. The first study has two sensor outputs, axial and radial. Assay indices start at 1; use the table to choose another sensor.

In [ ]:
study = wrapper.study(2)
pd.DataFrame([assay.model_dump() for assay in study.list_assays()])

In [ ]:
assay = study.assay(1)
runs = assay.list_runs()
print(f'{len(runs)} measurements available for this sensor')
pd.DataFrame([item.model_dump() for item in runs[:5]])

## Inspect a measurement

The wrapper's API calls these entries runs. The metadata describes the sampling rate, measured unit and operating conditions. Unit advisories are limited to known mismatches; no warning is not proof of scientific correctness.

In [ ]:
run = assay.run(runs[0].run_id)
overview = run.overview()
print('Run:', run.run_id)
print('File:', overview.raw_file_path)
print('Factors:', run.factor_values())
print('Measurement parameters:')
display(pd.DataFrame(overview.measurement_params))
print('Capabilities and resolved metadata:')
display(run.analysis_capabilities().model_dump())
print('Quantity/unit advisory:', run.quantity_unit_advisory())

## Load and plot the signal

This dataset contains raw files, so we explicitly request `file_type="raw"`. The wrapper converts the CSV time strings to seconds without resetting or rescaling them. Later measurements in this dataset start after zero; the timestamps do not by themselves prove synchronization.

In [ ]:
df, load_info = run.load_dataframe_with_meta(file_type='raw')
print('Rows:', len(df))
print('Load details:', load_info.model_dump())
display(df.head())
display(df.describe())

In [ ]:
from bokeh.io import output_notebook, show
output_notebook()
show(run.plot_timeseries())

## Compare a later measurement

To work with a zero-based axis for a particular analysis, create a separate copy. This does not alter the original CSV or the wrapper's returned values.

In [ ]:
later_run = assay.run(runs[-1].run_id)
later_df = later_run.load_dataframe(file_type='raw')
analysis_df = later_df.copy()
analysis_df['time_from_start'] = analysis_df['time'] - analysis_df['time'].iloc[0]
print('Later measurement:', later_run.run_id)
display(analysis_df.head())

## RMS and other features for one measurement

Use the wrapper to calculate features for this measurement. By default it prefers processed data if present, otherwise raw; the source is shown below. RMS includes any DC offset; no filtering or detrending is applied. Non-finite samples are excluded and counted. These features do not by themselves identify a fault.

In [ ]:
single_features = run.features()
display(pd.Series({key: value for key, value in single_features.items() if key != 'source'}, name='value'))
print('Source:', single_features['source']['resolved_file_type'], single_features['source']['file_path'])
print('Measured unit:', run.analysis_capabilities().measured_unit.value)

## RMS across the bearing's measurements

The wrapper can calculate features across every measurement of the selected sensor. This reads all its CSV files and can take a little while. The horizontal axis is measurement order, not a claim of exact global time. A change in RMS alone does not establish degradation or a fault.

In [ ]:
features = assay.lifecycle_features(file_type='raw', n_workers=1)
print('Measurement status counts:', features['status'].value_counts().to_dict())
display(features.loc[features['status'] != 'ok', ['run_id', 'file_path', 'error']])
display(features[['run_id', 'run_number', 'rms', 'peak2peak', 'kurtosis', 'crest_factor']].head())
selected = features.loc[features['run_id'] == run.run_id, 'rms']
print('Selected measurement RMS matches:', not selected.empty and np.isclose(selected.iloc[0], single_features['rms']))

In [ ]:
from bokeh.plotting import figure

unit = run.analysis_capabilities().measured_unit.value or 'unit unavailable'
rms_plot = figure(title='RMS across measurements', x_axis_label='Measurement number', y_axis_label=f'RMS ({unit})', width=850, height=350)
rms_plot.line(features['run_number'].to_numpy(), features['rms'].to_numpy(), line_width=2)
show(rms_plot)

The wrapper also calculates kurtosis (Fisher/excess kurtosis) and crest factor. Interpretation depends on signal conditioning, sampling and operating conditions; avoid treating a single feature as a fault label.

## Frequency-domain view

The spectrum shows signal content by frequency. The wrapper reads the explicit sampling rate from the measurement metadata (25.6 kHz for this dataset). A peak alone does not identify a bearing fault; further analysis and machine context are needed.

In [ ]:
show(assay.plot_frequency_domain(run_id=run.run_id, file_type='raw'))

## Prepare a feature table for modelling

Choose features explicitly with `feature_table()`. By default it stops if a selected feature or target is missing. Here we opt in to dropping incomplete measurements and inspect the accompanying report. The returned table includes source file information. This recalculates features for the selected sensor; no model is trained. RMS and peak-to-peak are examples, not universally recommended features.

In [ ]:
ml_data, excluded = assay.feature_table(
    label_column='fv_RUL',
    features=['rms', 'peak2peak'],
    file_type='raw',
    n_workers=1,
    on_incomplete='drop',
)
print(f'Included: {len(ml_data)}; excluded: {len(excluded)}')
display(excluded)  # run IDs and reasons; empty when nothing was excluded
display(ml_data.head())

The RUL target in this export is text such as `123 min`; the wrapper does not convert it to a number. Before modelling, verify the meaning and units, then convert deliberately. Measurements from the same bearing are related: randomly splitting individual measurements can leak information between training and test sets. Consider splitting by bearing when evaluating a model.

## Try it yourself

Change `wrapper.study(1)` to `wrapper.study(2)`, or `study.assay(1)` to `study.assay(2)`, and rerun the following cells. You can also select another entry from `runs`. This example only demonstrates exploration; it is not a validation of the dataset or a diagnosis.